# User Query Testing

This notebook helps a user run the saved model on a custom query after the pipeline has produced model artifacts.

Required artifacts: `data/experiments/run_config.json` and `data/experiments/logistic_model.joblib`.

In [3]:
from pathlib import Path
import pandas as pd
import sys

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

from test_model import prepare_runtime
from ranking_system.features import build_inference_frame
from ranking_system.modeling import score_rows
from ranking_system.pipeline import blend_hybrid_scores

query = "best budget gaming laptop"
top_k = 5
candidate_k = 20

_, corpus, bm25_index, feature_assets, trained_model, hybrid_alpha = prepare_runtime(ROOT / "data/experiments")

candidate_doc_ids = [doc_id for doc_id, _ in bm25_index.top_k(query, candidate_k)]
inference_df = build_inference_frame(
    query_id="notebook_query",
    query=query,
    candidate_doc_ids=candidate_doc_ids,
    corpus=corpus,
    bm25_index=bm25_index,
    feature_assets=feature_assets,
)

scored_df = score_rows(trained_model, inference_df)
hybrid_df = blend_hybrid_scores(scored_df, hybrid_alpha)
ranked = hybrid_df.sort_values("hybrid_score", ascending=False).head(top_k).copy()
ranked["title"] = ranked["doc_id"].map(lambda doc_id: corpus[doc_id].get("title", ""))
ranked["text_preview"] = ranked["doc_id"].map(lambda doc_id: corpus[doc_id].get("text", "").replace("\n", " ")[:220])
ranked[["doc_id", "title", "text_preview", "bm25", "score", "hybrid_score"]]


BM25 tokenizing docs: 100%|██████████| 500000/500000 [00:07<00:00, 69282.85doc/s] 


,doc_id,title,text_preview,bm25,score,hybrid_score
15,1072536,,6 GB of DDR3 Memory. 750GB Hard Drive. The mod...,15.181382,0.913980,1.000000
6,1410210,,HP OMEN 17 17.3'' UHD 4K Gaming and Business L...,16.741297,0.733603,0.801367
4,1260086,,"âDesktop processors, Pascal GPUs and a stunn...",18.074509,0.606782,0.661712
19,1285321,,1 SolvedWhich laptop is better: 1st generation...,14.853544,0.604103,0.658762
17,1198320,,Gaming on the Surface Book: What you need to k...,14.949411,0.511558,0.556851
